# BART (Short Summary)

BART is a:

```text
denoising encoder-decoder Transformer
```

trained by:

```text
corrupt text
→ reconstruct original text
```

---

# Architecture

BART combines:

| Part | Similar To |
|---|---|
| Encoder | BERT |
| Decoder | GPT |

So BART can:
- understand text
- generate text

---

# Example

Original:

```text
The patient received medicine at the hospital
```

Corrupted:

```text
The patient <mask> at hospital
```

Target:

```text
The patient received medicine at the hospital
```

---

# Corruption Methods

BART uses:
- token masking
- token deletion
- span masking
- sentence shuffling

---

# Why Important

BART works very well for:
- summarization
- translation
- text generation
- question answering

because it combines:
- bidirectional understanding
- autoregressive generation

---

# Final Insight

## BERT

```text
understand text
```

## GPT

```text
generate text
```

## BART

```text
understand + generate
```

In [1]:
# pip install transformers torch sentencepiece

import torch
from transformers import BartTokenizer, BartForConditionalGeneration

# -----------------------------
# 1. Load pretrained BART
# -----------------------------

MODEL_NAME = "facebook/bart-base"

tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)
model = BartForConditionalGeneration.from_pretrained(MODEL_NAME)

# -----------------------------
# 2. Example text
# -----------------------------

text = """
The patient was diagnosed with diabetes and received medicine at the hospital.
The doctor recommended continued treatment and regular monitoring.
"""

# -----------------------------
# 3. Tokenize input
# -----------------------------

inputs = tokenizer(
    text,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

# -----------------------------
# 4. Generate summary
# -----------------------------

model.eval()

with torch.no_grad():
    summary_ids = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=40,
        min_length=5,
        num_beams=4,
        early_stopping=True
    )

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("Summary:")
print(summary)

D:\anaconda3\envs\web_lumplt\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
D:\anaconda3\envs\web_lumplt\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shiri\.cache\huggingface\hub\models--facebook--bart-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see t

Summary:
The patient was diagnosed with diabetes and received medicine at the hospital.The doctor recommended continued treatment and regular monitoring.


# BART Fine-Tuning for Sequence Classification

For sequence classification tasks, BART feeds the same input sentence into both the encoder and decoder. The encoder processes the sentence using bidirectional self-attention, allowing every token to attend to all other tokens and build a full contextual understanding of the input.

Example input:

```text
The movie was very emotional and inspiring.
```

Encoder input:

```text
The movie was very emotional and inspiring.
```

Decoder input:

```text
<s> The movie was very emotional and inspiring.
```

where:

```text
<s>
```

is the start token.

The decoder processes the sequence autoregressively while also attending to the encoder outputs. After decoding, the hidden state of the final decoder token is used as the sentence-level representation.

For example, the final decoder token:

```text
inspiring
```

produces a hidden representation containing information about the entire sentence.

This representation is passed into a linear classification layer:

```text
Final Decoder Hidden State
        ↓
Linear Layer
        ↓
Positive Sentiment
```

This is similar to the `[CLS]` token in BERT, but instead of using a special encoder classification token, BART uses the final decoder hidden state for sequence classification.

# BART Fine-Tuning for Token Classification

For token classification tasks, BART uses the decoder hidden representation for each token individually instead of using only the final decoder token.

The complete document is fed into both the encoder and decoder. The encoder processes the full input using bidirectional attention, while the decoder generates contextual token representations autoregressively.

Example input:

```text
John lives in Seattle.
```

Encoder input:

```text
John lives in Seattle.
```

Decoder input:

```text
<s> John lives in Seattle.
```

After decoding, every decoder token has its own hidden representation:

| Token | Decoder Hidden State |
|---|---|
| John | h1 |
| lives | h2 |
| in | h3 |
| Seattle | h4 |

These token-level decoder representations are then passed into a classifier separately.

For example, in question answering (SQuAD), the model predicts:
- answer start token
- answer end token

Suppose the question is:

```text
Where does John live?
```

The model may classify:

```text
Seattle
```

as the answer token span.

Flow:

```text
Input Document
      ↓
Encoder
      ↓
Decoder
      ↓
Hidden State for Each Token
      ↓
Token-Level Classifier
      ↓
Predicted Labels
```

Unlike sequence classification, where only the final decoder hidden state is used, token classification uses:

```text
decoder hidden state for every token
```

to make predictions for each word individually.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# -----------------------------
# 1. Tiny vocabulary
# -----------------------------

vocab = {
    "<pad>": 0,
    "<bos>": 1,
    "<eos>": 2,
    "<mask>": 3,
    "the": 4,
    "patient": 5,
    "received": 6,
    "medicine": 7,
    "at": 8,
    "hospital": 9,
}

id_to_word = {v: k for k, v in vocab.items()}

PAD = vocab["<pad>"]
BOS = vocab["<bos>"]
EOS = vocab["<eos>"]
MASK = vocab["<mask>"]

# -----------------------------
# 2. BART-style denoising data
# -----------------------------

# Original text:
# the patient received medicine at hospital

# Corrupted input:
# the patient <mask> at hospital

encoder_input_ids = torch.tensor([
    [vocab["the"], vocab["patient"], MASK, vocab["at"], vocab["hospital"], EOS]
])

# Decoder input starts with BOS
decoder_input_ids = torch.tensor([
    [BOS, vocab["the"], vocab["patient"], vocab["received"], vocab["medicine"], vocab["at"]]
])

# Target is the original clean sentence
target_ids = torch.tensor([
    [vocab["the"], vocab["patient"], vocab["received"], vocab["medicine"], vocab["at"], vocab["hospital"]]
])

# -----------------------------
# 3. Tiny BART model
# -----------------------------

class TinyBART(nn.Module):
    def __init__(self, vocab_size, d_model=32, num_heads=4, num_layers=2):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=num_heads,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=64,
            batch_first=True
        )

        self.lm_head = nn.Linear(d_model, vocab_size)

    def causal_mask(self, size):
        return torch.triu(torch.ones(size, size), diagonal=1).bool()

    def forward(self, encoder_input_ids, decoder_input_ids):
        enc = self.embedding(encoder_input_ids)
        dec = self.embedding(decoder_input_ids)

        tgt_len = decoder_input_ids.size(1)
        tgt_mask = self.causal_mask(tgt_len)

        output = self.transformer(
            src=enc,
            tgt=dec,
            tgt_mask=tgt_mask
        )

        logits = self.lm_head(output)

        return logits

# -----------------------------
# 4. Train
# -----------------------------

model = TinyBART(vocab_size=len(vocab))

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)

for epoch in range(300):
    logits = model(encoder_input_ids, decoder_input_ids)

    loss = loss_fn(
        logits.reshape(-1, len(vocab)),
        target_ids.reshape(-1)
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}, Loss={loss.item():.4f}")

# -----------------------------
# 5. Generate reconstructed text
# -----------------------------

model.eval()

generated = [BOS]

for _ in range(6):
    decoder_ids = torch.tensor([generated])

    with torch.no_grad():
        logits = model(encoder_input_ids, decoder_ids)

    next_token = logits[:, -1, :].argmax(dim=-1).item()

    if next_token == EOS:
        break

    generated.append(next_token)

words = [
    id_to_word[i]
    for i in generated
    if i not in [BOS, EOS, PAD]
]

print("\nCorrupted Input:")
print("the patient <mask> at hospital")

print("\nReconstructed Output:")
print(" ".join(words))

# T5 vs BART

Both T5 and BART are:

```text
encoder-decoder Transformers
```

used for:
- summarization
- translation
- QA
- text generation

BUT their training objectives differ.

---

# Main Difference

| T5 | BART |
|---|---|
| text-to-text framework | denoising autoencoder |
| span corruption | arbitrary corruption |
| predicts missing spans | reconstructs full text |
| unified task format | denoising reconstruction |

---

# T5 Pretraining

T5 corrupts spans of text.

Example:

Original:

```text
The patient received medicine at hospital
```

Input:

```text
The patient <extra_id_0> hospital
```

Target:

```text
<extra_id_0> received medicine at
```

T5 predicts:
- only missing spans.

---

# BART Pretraining

BART corrupts text and reconstructs the ENTIRE original sentence.

Example:

Input:

```text
The patient <mask> hospital
```

Target:

```text
The patient received medicine at hospital
```

BART predicts:
- full clean sentence.

---

# Corruption Methods

## T5

Mainly:
- span masking

---

# BART

Uses many noise types:
- token masking
- deletion
- sentence shuffling
- text infilling
- document rotation

---

# Task Format

## T5

Everything becomes:

```text
text → text
```

Examples:

```text
summarize: ...
translate English to French: ...
question: ...
```

---

# BART

More traditional fine-tuning:
- summarization head
- classification head
- QA head

---

# Architecture Difference

| Feature | T5 | BART |
|---|---|
| Encoder-decoder | YES | YES |
| Relative position bias | YES | NO |
| Absolute position embeddings | NO | YES |
| Decoder causal masking | YES | YES |

---

# Position Embedding

## T5

Uses:

```text
relative position bias
```

---

# BART

Uses:

```text
learned absolute positional embeddings
```

---

# Training Objective Difference

## T5

```text
recover missing spans
```

---

# BART

```text
reconstruct entire corrupted sequence
```

---

# Simple Intuition

## T5

```text
fill in missing pieces
```

---

# BART

```text
repair damaged text
```

---

# Final Insight

## T5

```text
unified text-to-text Transformer
```

focused on:
- flexible NLP tasks.

---

# BART

```text
denoising seq2seq Transformer
```

focused on:
- powerful text reconstruction and generation.

# Models Compared in the BART Paper

The BART paper compares several pretraining objectives to understand which learning strategy works best for NLP tasks.

---

# 1. Language Model (GPT-style)

```text
left-to-right next-token prediction
```

Example:

Input:

```text
The patient received
```

Target:

```text
medicine
```

Characteristics:
- decoder-only Transformer
- causal masking
- autoregressive generation

Equivalent to:
- GPT
- BART decoder without encoder cross-attention

---

# 2. Permuted Language Model (XLNet-style)

Instead of predicting tokens left-to-right, the model predicts tokens in:

```text
random order
```

Example:

Sentence:

```text
The patient received medicine
```

Possible prediction order:

```text
medicine → The → received → patient
```

Goal:
- learn bidirectional context
- avoid fixed left-to-right bias

Based on:
- XLNet

---

# 3. Masked Language Model (BERT-style)

Random tokens are masked and predicted independently.

Example:

Input:

```text
The patient [MASK] medicine
```

Target:

```text
received
```

Characteristics:
- bidirectional encoder
- predicts masked tokens only
- no autoregressive generation

Based on:
- BERT

---

# 4. Multitask Masked Language Model (UniLM-style)

Uses different attention masks during training:
- left-to-right
- right-to-left
- fully bidirectional
- partially masked

Goal:
- make one model learn multiple attention behaviors.

Based on:
- UniLM

---

# 5. Masked Seq-to-Seq (MASS-style)

Large spans are masked and the decoder predicts ONLY the masked part.

Example:

Input:

```text
The patient [MASK]
```

Target:

```text
received medicine at hospital
```

Characteristics:
- encoder-decoder training
- predicts missing spans
- similar to seq2seq generation

Based on:
- MASS

---

# Why BART Compared Them

BART wanted to show:

```text
one general denoising seq2seq objective
can outperform many specialized objectives
```

---

# Key Insight

| Model Type | Main Idea |
|---|---|
| GPT | next-token prediction |
| BERT | masked token prediction |
| XLNet | permutation prediction |
| MASS | masked seq2seq |
| BART | reconstruct corrupted text |

---

# Final Insight

BART unified ideas from:
- GPT generation
- BERT bidirectional understanding
- MASS seq2seq masking

into one:

```text
denoising encoder-decoder Transformer
```